# 19 — The De Palma limitation: when quantum optimal transport bounds the algorithm

Turn the geometry around. Across this module you used quantum optimal transport to *measure* — to read the coupling between two systems off a density matrix. Here you meet a result where the very same geometry becomes a *proof technique*: the quantum Wasserstein distance lower-bounds what shallow variational quantum algorithms can achieve, drawing a provable line around a whole family of near-term quantum methods.

**Prerequisites:** `16_capstone_synthesis` (and the quantum-Wasserstein thread of `03_trevisan_taxonomy`, `09_the_qot_zoo`).

**What you'll be able to do**
- Explain the turn from QOT-as-*measurement* to QOT-as-*proof-technique*: the same transport geometry that read coupling now bounds a computation.
- State the **De Palma, Marvian, Rouzé & Stilck França (2023)** limitation theorem *accurately* — a limitation on *shallow, local* variational circuits (VQE / QAOA), conditioned on depth and locality, not a blanket claim that "VQE does not work".
- Give the intuition in one line: a shallow local circuit moves its output only a bounded quantum-$W_1$ distance from its product reference, and a Lipschitz cost can therefore change by only so much — which lower-bounds the energy such a circuit can reach.
- Situate the module's coupling SDP inside the **broader QOT taxonomy** — the quantum-$W_1$ (Lipschitz / observable) family is a *different corner* from the coupling object you built, and it is the corner this theorem lives in.
- Name three honest **open problems** at this frontier.

You arrive having closed the applied arc. In `16` you held the capstone's two truths at once — a real, controlled case where a richer-embedding QOT measure beat the phase-locking value, priced honestly and bounded precisely. That was QOT pointed *at data*, as an instrument. This notebook points the same machinery the other way: at *computation itself*. It asks not "what does this state's coupling look like?" but "how far can a shallow quantum circuit even travel?" — and answers it with a transport bound. No new code is required to follow the argument; one small cartoon makes the geometry visible. This is mostly a result to understand, state precisely, and place on the map.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS

np.random.seed(42)
viz.use_course_style()


## 1. From measuring coupling to bounding algorithms

Every use of quantum optimal transport in this module has been *constructive*. You built the coupling SDP (`04`) to turn two density matrices into a transport cost; you added an entropy bonus (`06`–`08`) to make it scale; you embedded signals into states (`10`) and read the coupling between two oscillators off the resulting joint density (`11`–`16`). In every case QOT was an **instrument**: feed it states, read out a number that means *how far apart* or *how coupled*.

There is a second way to use a geometry, and it is the subject of this notebook. A distance does not only *report* how far two objects are — it *constrains* how far you can get from a starting point in a bounded number of moves. If each step of a process can increase a distance by at most some amount, then after a few steps you are still provably close to where you began. That is a statement about *reachability*, and it is exactly the kind of statement that produces a **limitation theorem**: if the good answers all live far away in the distance, and your process cannot travel far, then your process cannot reach the good answers.

This is the turn De Palma, Marvian, Rouzé & Stilck França (2023) make. The geometry is the **quantum Wasserstein distance of order 1** — a quantum optimal-transport distance, a cousin of the coupling object this module built (we place the two precisely in §3). The "process" is a **variational quantum algorithm**: a parametrised circuit $U(\theta)$ applied to a simple reference state, tuned to minimise the energy $\langle \psi(\theta) | H | \psi(\theta) \rangle$ of some cost Hamiltonian $H$. The limitation is that a *shallow, local* such circuit cannot move its output far in quantum-$W_1$ from its product reference — and so, for cost Hamiltonians that are smooth (Lipschitz) in that distance, it cannot reach an energy far below the reference's. The transport geometry the course built to *measure* coupling is, in this paper, the instrument that *proves a boundary* on quantum computation.

Hold the shift in one sentence: **the same quantum-Wasserstein geometry is constructive when you point it at data and restrictive when you point it at an algorithm** — and a framework worth trusting is one sharp enough to do both.


## 2. The De Palma et al. theorem

Let us set the stage carefully, because the value of this result is in its precision — overstated, it becomes a myth; stated exactly, it is a genuine boundary.

### The object: a variational quantum algorithm

A **variational quantum algorithm** (VQA) — the family that includes the *variational quantum eigensolver* (VQE) and the *quantum approximate optimisation algorithm* (QAOA) — prepares a trial state by applying a parametrised circuit to a fixed, simple reference:

$$ |\psi(\theta)\rangle \;=\; U(\theta)\,|\psi_0\rangle, \qquad U(\theta) = U_L(\theta_L)\cdots U_1(\theta_1), $$

where $|\psi_0\rangle$ is typically a **product state** (each qubit prepared independently, e.g. $|0\cdots 0\rangle$ or $|+\cdots+\rangle$), and each layer $U_\ell$ is built from **local** gates (each gate touches only a few neighbouring qubits). The algorithm tunes $\theta$ to minimise the energy $\mathrm{tr}\!\big(H\,|\psi(\theta)\rangle\langle\psi(\theta)|\big)$ of a cost Hamiltonian $H$ — for QAOA, $H$ encodes a combinatorial problem such as Max-Cut. Two numbers describe the circuit's reach: its **depth** $L$ (how many layers) and its **locality** (how many qubits each gate touches, hence how fast information can spread).

### The geometry: a shallow local circuit travels only a bounded $W_1$ distance

Here is the engine of the proof, and the place quantum optimal transport enters. The quantum-$W_1$ distance (De Palma, Marvian, Trevisan & Lloyd, 2021 — §3) comes equipped with a **contraction property under local channels**: each layer of a local circuit can increase the quantum-$W_1$ distance between two states by at most a bounded factor, set by the locality of its gates. Iterating over $L$ layers, the output of a depth-$L$ local circuit sits within a **bounded quantum-$W_1$ ball** around the image of the reference state — a ball whose radius grows with depth and locality, and no faster. Equivalently, through the light-cone picture: in shallow depth, each qubit's reduced state depends only on the few qubits inside its causal cone, so the output stays close — in transport distance — to a product state.

This is the precise sense in which "shallow circuits cannot go far": *far* is measured in quantum-$W_1$, and the circuit's depth caps the radius it can reach.

### The bound: a Lipschitz cost cannot change more than the distance allows

The quantum-$W_1$ distance is, by construction, the **dual of a quantum Lipschitz constant** on observables (§3). For any observable $O$ with quantum Lipschitz constant $\|O\|_{L}$ and any two states $\rho, \sigma$,

$$ \big|\,\mathrm{tr}(O\rho) - \mathrm{tr}(O\sigma)\,\big| \;\le\; \|O\|_{L}\;\cdot\; W_1(\rho, \sigma). $$

Read it slowly: *if two states are close in $W_1$, then every Lipschitz observable takes nearly the same expectation on both*. Now chain the two facts. The output $\rho_\theta = |\psi(\theta)\rangle\langle\psi(\theta)|$ of a shallow local circuit is within a bounded $W_1$ distance of (the image of) the product reference. So for a cost Hamiltonian $H$ with a finite quantum Lipschitz constant, the achievable energy $\mathrm{tr}(H\rho_\theta)$ cannot fall more than $\|H\|_L \cdot W_1(\rho_\theta, \text{reference})$ below the reference's energy. **The transport bound on reachability becomes a lower bound on the energy a shallow VQE/QAOA can reach.** Tuning $\theta$ cannot beat what the geometry forbids.

### State it accurately — and state what it does *not* say

> **De Palma, Marvian, Rouzé & Stilck França (2023).** Using quantum optimal transport — the quantum-$W_1$ distance, its Lipschitz duality, and concentration inequalities for Lipschitz observables — they prove that *shallow, local* variational quantum circuits face provable performance limitations. In the **noiseless** regime, the output of a low-depth local circuit stays within a bounded $W_1$ distance of its product reference, lower-bounding the energy (cost) it can achieve for Lipschitz cost Hamiltonians. In the **noisy** regime (local depolarising noise at rate $p$), they sharpen this dramatically: at depths $L = \mathcal{O}(1/p)$ the circuit output **concentrates**, and it becomes exponentially unlikely to outperform efficient *classical* algorithms on problems such as Max-Cut.

The honest reading carries three guardrails:

- **It is conditional on depth and locality.** The limitation bites for *shallow* circuits made of *local* gates — exactly the near-term (NISQ) regime the paper targets. A deep enough circuit can prepare a state arbitrarily far in $W_1$; the theorem does not touch it. The depth is the dial.
- **It is not "VQE does not work".** It is a precise no-go for a *regime*: low depth + locality + a Lipschitz cost. It says naive "exponential quantum advantage from a shallow variational circuit" claims, for that class of cost, cannot hold — not that variational methods are useless everywhere.
- **The cost must be Lipschitz in the quantum-$W_1$ sense** for the bound to apply. The Lipschitz constant $\|H\|_L$ sets the scale of the bound; the result speaks to the broad class of local cost Hamiltonians (such as Max-Cut) that have a controlled one.

That precision is the result. It is a *boundary drawn by a distance*: the good states are far away in quantum-$W_1$, shallow local circuits cannot travel that far, so they cannot reach them — proved with the same optimal-transport geometry this course built to measure coupling.


A single picture makes the reachability argument concrete. It is a **cartoon**, not a measurement — the radii are drawn to illustrate the *shape* of the statement (a $W_1$ ball whose radius grows with depth, while the target sits outside the shallow ball), not computed from any circuit. We label it as such, honestly.


In [ ]:
# A schematic of the theorem's geometry: in quantum-W1 space, a depth-L local circuit
# reaches a bounded ball around its product reference; the ball's radius grows with L.
# The good (low-energy) target sits FAR away -- outside the shallow ball. These radii
# are illustrative (a cartoon of the statement), NOT computed from a circuit.
depths = [1, 2, 3]
radii = [1.0, 1.9, 2.7]  # radius grows with depth, sub-linearly -- schematic only
ball_shades = [0.16, 0.11, 0.07]  # outer balls drawn fainter

fig, ax = plt.subplots(figsize=(8.5, 6.5))
ax.set_aspect("equal")

# The product reference: the centre every shallow circuit expands from.
ref = np.array([0.0, 0.0])

# Nested reachable balls, one per depth (drawn outermost-first so labels stay legible).
for L, r, shade in sorted(zip(depths, radii, ball_shades), key=lambda t: -t[1]):
    ax.add_patch(plt.Circle(ref, r, facecolor=COLORS["quantum"], alpha=shade,
                            edgecolor=COLORS["quantum"], linewidth=1.6, zorder=1))
    ax.annotate(rf"depth $L={L}$", (ref[0], ref[1] + r), xytext=(0, 6),
                textcoords="offset points", ha="center", va="bottom",
                fontsize=10.5, color=COLORS["text"], zorder=4)

# The reference state itself.
ax.scatter(*ref, s=140, color=COLORS["source"], edgecolor="white", linewidth=1.5,
           zorder=5)
ax.annotate("product reference\n$|\\psi_0\\rangle$", ref, xytext=(-14, -22),
            textcoords="offset points", ha="center", va="top", fontsize=10.5,
            color=COLORS["text"], zorder=5)

# The low-energy target -- placed beyond the deepest shallow ball.
target = np.array([4.3, 1.1])
ax.scatter(*target, s=220, marker="*", color=COLORS["highlight"], edgecolor="white",
           linewidth=1.2, zorder=6)
ax.annotate("low-energy target\n(ground state of $H$)", target, xytext=(8, 10),
            textcoords="offset points", ha="left", va="bottom", fontsize=10.5,
            color=COLORS["text"], zorder=6)

# The W1 gap the shallow circuit cannot close.
gap_edge = ref + (target - ref) / np.linalg.norm(target - ref) * radii[-1]
ax.annotate("", xy=target, xytext=gap_edge,
            arrowprops=dict(arrowstyle="<->", color=COLORS["negative"], lw=1.8))
mid = (gap_edge + target) / 2
ax.annotate("unreachable\n$W_1$ gap", mid, xytext=(4, -30), textcoords="offset points",
            ha="center", va="top", fontsize=10, style="italic",
            color=COLORS["negative"], zorder=6)

ax.set_xlim(-3.2, 7.6)
ax.set_ylim(-3.4, 4.6)
ax.set_xticks([])
ax.set_yticks([])
ax.grid(visible=False)
ax.set_xlabel(r"quantum-$W_1$ distance from the reference  (schematic axes)")
ax.set_title("Cartoon: the reachable $W_1$ ball grows with depth — the target sits outside",
             pad=12)
fig.tight_layout()
plt.show()


**Read the figure.** This is the theorem's logic drawn in transport space, and nothing more — the numbers on the axes are illustrative. The periwinkle dot at the centre is the circuit's **product reference** $|\psi_0\rangle$, the simple state every variational circuit starts from. The nested blue discs are the **reachable sets** at increasing depth: a depth-$1$ local circuit can move its output anywhere inside the smallest disc (a bounded quantum-$W_1$ radius), depth $2$ reaches the next disc out, depth $3$ the next — the radius grows with depth, and the contraction-coefficient bound is what caps how fast. The rose star is a **low-energy target**, the kind of state a good answer would require, drawn deliberately *outside* the deepest shallow ball. The coral arrow is the **$W_1$ gap the shallow circuit cannot cross**: because every Lipschitz cost changes by at most $\|H\|_L \cdot W_1$, a target this far in $W_1$ has an energy the shallow circuit provably cannot reach. Make the circuit deeper and the discs grow until they finally swallow the star — which is exactly why the theorem is a statement *about depth*, not a verdict against variational methods as such. The cartoon shows the shape of the bound; the paper supplies the rigorous radii.


## 3. The broader QOT taxonomy — and where this theorem lives

This module committed, deliberately and from `03`, to **one** quantum optimal-transport object: the **coupling SDP**, a bipartite density matrix $\rho_{AB} \succeq 0$ with prescribed partial-trace marginals, minimising $\mathrm{tr}(C\,\rho_{AB})$. It is the closest analogue of the Kantorovich linear program, and it carried the entropic regularisation and the information-geometric bridges of the course. But it is *one corner* of a larger landscape — and crucially, **it is not the corner this theorem lives in**. The De Palma et al. limitation runs on a *different* QOT object, the quantum-$W_1$ distance, and seeing why is the point of placing them side by side.

Trevisan's 2022 survey is the map. The table situates the families, with the one carrying §2's proof marked.

| Family | Core idea | Relation to this module |
|---|---|---|
| **Coupling SDP** (De Palma–Trevisan 2021; Cole et al. 2023) | bipartite $\rho_{AB} \succeq 0$ with marginals; minimise $\mathrm{tr}(C\,\rho_{AB})$ | **the object this module built** (`04`–`16`); a static distance *between two states* |
| **Entropic / quantum Sinkhorn** (Peyré–Chizat–Vialard–Schmitzer 2019) | the coupling SDP with a von-Neumann-entropy regulariser; matrix-exp Gibbs kernel | built in `06`–`08`; the Amari bridge survives the lift |
| **Quantum $W_1$ / Lipschitz** (De Palma–Marvian–Trevisan–Lloyd 2021) | a quantum Lipschitz constant on multi-qubit observables; $W_1$ is its dual | **the tool of §2's proof** — a metric *on many-body states*, with concentration and contraction inequalities; *not* generally equal to the coupling-SDP value |
| **Channel-based** (De Palma–Trevisan 2021, other version) | a Wasserstein-like distance from CPTP-channel actions via Stinespring dilations | survey context; no explicit coupling object |
| **Dynamic / Carlen–Maas** (Carlen–Maas 2014) | replace the continuity equation by a Lindblad-type generator; a $W_2$ on the *dynamics* | survey context; ties to quantum gradient flow |

**These families are not interchangeable.** They agree on classical (commuting) states and on Gaussians — that shared commuting limit is the consistency principle of `03` — but off those special cases they answer different questions and return different numbers. The coupling SDP is built to be a *distance between two given states* and reads off a transport cost; the quantum-$W_1$ family is built around a *Lipschitz / observable* structure on many-body systems and comes with the *concentration* and *contraction* machinery that §2 needs. That machinery — bounding how far a state can move under a shallow circuit, and how much a Lipschitz observable can therefore change — is precisely why the limitation theorem is stated in quantum-$W_1$ and not in the coupling SDP. The lesson is the one the course has repeated: there is no single "*the*" quantum optimal transport; the right object is the one whose structure matches the question, and a *limitation-by-concentration* question calls for the Lipschitz / $W_1$ corner of the map.


## 4. Open problems at this frontier

A frontier is honest about what it does not yet know. Three questions sit open right where this module's machinery and this notebook's theorem meet — each a genuine place to start, not a settled matter.

1. **How tight is the depth threshold, and where exactly does advantage survive?** The theorem bounds *shallow, local* circuits; deeper or less-local circuits escape it. The sharp boundary — the precise depth, locality, and cost-Hamiltonian structure at which a provable quantum advantage can still exist — is not fully mapped. The noisy-regime threshold $L = \mathcal{O}(1/p)$ is dramatic but specific to local depolarising noise and problems like Max-Cut; the full phase diagram of *which* problems, depths, and noise models admit advantage is open.

2. **What is the right QOT object for a given limitation, and how do the families relate quantitatively?** The coupling SDP this module built and the quantum-$W_1$ family of §2 agree on commuting states but differ off them, and the *quantitative* relationship between them on general many-body states is not fully characterised. A cleaner dictionary between the QOT families — when a bound in one transfers to another — would let results proved in one corner travel to the others.

3. **Does QOT-based coupling carry over to real, noisy, finite data — the hyperscanning question?** The capstone (`11`–`16`) demonstrated, on a controlled synthetic dyad, a case where a richer-embedding QOT measure beats the phase-locking value — priced honestly, and without claiming uniqueness over classical higher-order statistics. Whether that advantage survives on *real* EEG hyperscanning data, with its noise, nonstationarity, and finite samples, is untested and open. The concentration phenomena of this notebook are a caution worth carrying into that question: high-dimensional quantum objects estimated from limited data can concentrate, and a frontier result about *circuits* is a reminder to ask the same of *estimators*.

These are not the only open problems — but they are the ones this course's machinery, and this notebook's theorem, are best placed to attack.


## Your turn

These are reasoning prompts — no code is required. They consolidate the theorem and its place in the map.

### Warm-up

In one or two sentences each, say *why depth is the dial*. First: explain, using the nested-ball cartoon of §2, why a deeper circuit escapes the limitation while a shallow one does not — what grows, and what that lets the circuit reach. Then: state, in your own words, the one property of a cost Hamiltonian $H$ that the bound $|\mathrm{tr}(H\rho_\theta) - \mathrm{tr}(H\,\text{ref})| \le \|H\|_L \cdot W_1$ requires, and what would happen to the bound if that property failed (if $\|H\|_L$ were unbounded).

### Go further

A colleague summarises this notebook as "quantum optimal transport proves that VQE does not work." Write the two- or three-sentence correction you would send back. Name the *two conditions* the theorem actually requires (think depth and locality), name the *regime* it targets (near-term / NISQ), and state precisely what it forbids — a class of claims about *shallow* circuits — versus what it leaves entirely open.

### Relate it to the capstone

The capstone (`16`) used a QOT measure as an *instrument* pointed at data; this notebook used a QOT distance as a *proof technique* pointed at an algorithm. In a short paragraph, contrast the two uses: in the capstone, the *embedding* was the dial that set what the coupling measure could see; here, the *circuit depth* is the dial that sets how far the algorithm can travel. What is common to both stories about *which* quantum-optimal-transport object was the right tool for the question — the coupling SDP for measuring a fixed pair of states, versus the quantum-$W_1$ / Lipschitz object for bounding reachability under a process?


## What you built

You have crossed from using quantum optimal transport to understanding it as a research instrument with two faces — and you did it without softening either one.

- You made the turn from **QOT-as-measurement to QOT-as-proof-technique**: the same transport geometry that read coupling off a density matrix throughout this module becomes, in the hands of De Palma et al., the instrument that *bounds* a computation.
- You can state the **De Palma, Marvian, Rouzé & Stilck França (2023)** theorem *accurately*: a limitation on *shallow, local* variational circuits (VQE / QAOA), conditioned on depth and locality and a Lipschitz cost — a shallow circuit moves only a bounded quantum-$W_1$ distance from its product reference, which lower-bounds the energy it can reach (and, with noise at depth $\mathcal{O}(1/p)$, makes beating classical baselines exponentially unlikely). You can say what it does *not* claim: not that variational methods are useless.
- You placed this module's **coupling SDP inside the broader QOT taxonomy** and saw that the theorem lives in a *different* corner — the quantum-$W_1$ / Lipschitz family, whose concentration and contraction structure is what the proof needs — reinforcing the course's refrain that the right QOT object is the one matched to the question.
- You named three honest **open problems**: the sharp depth/advantage boundary, the quantitative relations among QOT families, and whether QOT coupling survives on real hyperscanning data.

Take the measure of this. You now hold quantum optimal transport as *both* a measurement tool — the coupling measure you demonstrated and bounded in the capstone — *and* a proof technique at the research frontier, a geometry sharp enough to draw a provable line around a family of quantum algorithms. Seeing one mathematics serve both roles, and stating each precisely, is what it means to understand a tool rather than merely run it.

## What's next

One notebook remains. In `20_frontier_synthesis` the whole course comes back into one frame — the greatest hits replayed and the completed classical ↔ quantum dictionary read in full — closing the thirty-hour arc from "moving a pile of mass" to a quantum optimal transport you can both compute with and reason about. Bring this notebook's double vision with you: it is the last row the dictionary needs.


## References

- G. De Palma, M. Marvian, C. Rouzé & D. Stilck França, "Limitations of variational quantum algorithms: a quantum optimal transport approach", *PRX Quantum* **4**, 010309 (2023). DOI:10.1103/PRXQuantum.4.010309 — arXiv:2204.03455. The frontier result of this notebook: quantum optimal transport (the quantum-$W_1$ distance, its Lipschitz duality, and concentration inequalities) lower-bounds what shallow, local variational quantum algorithms can achieve.
- G. De Palma, M. Marvian, D. Trevisan & S. Lloyd, "The quantum Wasserstein distance of order 1", *IEEE Transactions on Information Theory* **67**, 6627–6643 (2021). DOI:10.1109/TIT.2021.3076442 — arXiv:2009.04469. Defines the quantum-$W_1$ distance via a quantum Lipschitz constant on observables (its dual), and proves the Marton transportation inequality, the Gaussian concentration inequality for Lipschitz observables, and the contraction-coefficient bounds for shallow circuits that §2's argument rests on.
- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091 — the survey whose taxonomy organises §3; situates the coupling SDP, the quantum-$W_1$ / Lipschitz family, the channel-based and dynamic formulations on one map.
- G. De Palma & D. Trevisan, "Quantum optimal transport with quantum channels", *Annales Henri Poincaré* **22**, 3199–3234 (2021). DOI:10.1007/s00023-021-01042-3 — the coupling and channel-based formulations that anchor the taxonomy's first and fourth rows.

**Previous:** `notebooks/04_QuantumOptimalTransport/18_what_no_classical_data_embedding_can_buy.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/20_frontier_synthesis.ipynb`
